# 1. Data cleaning and pre-processing

This notebook walks through the steps of:
1. Loading the raw 2016 ANES data
2. Cleaning text fields

Input:\
`data/ANES_2016.csv` : raw data

Output:\
`data/ANES_2016_CLEANED.txt` : a tab-seperated file with `_clean` fields for every text response. 

In [1]:
# helpful packages
import pandas as pd
import numpy as np

# language models
import spacy
from spacy.tokens import Doc
import contextualSpellCheck

# language clustering
from gensim.models import KeyedVectors
import umap
import hdbscan

# network analysis
import networkx as nx
import netstats as ns

In [54]:
np.random.seed(42)

import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# 1. Load ANES 2016 data

In [6]:
#### load raw data
filename = 'data/ANES_2016.csv'

df = pd.read_csv(filename)

print(f'Data includes {len(df)} observations on {len(df.columns)} variables.')
print(f'\nRecorded variables are:\n{", ".join(df.columns)}')

Data includes 4270 observations on 21 variables.

Recorded variables are:
V160001, V160001_orig, age, female, education, leftright, politicalinterest, politicalparticipation, TIPI_extraversion, TIPI_agreeableness, TIPI_conscientiousness, TIPI_emotionalstability, TIPI_openness, V161069, V161072, V161075, V161078, V161098, V161101, V161104, V161106


In [7]:
#### Identify key variables
# note: V160001 and V160001_orig are unique respondent ids

# Questions with free-response answers. This will be the core of the analysis
text_qs = ['V161069', # PRE: Text- What is it that R likes about Democratic Pres cand
           'V161072', # PRE: Text- What is it that R dislikes about Democratic Pres cand
           'V161075', # PRE: Text- What is it that R likes about Republican Pres cand
           'V161078', # PRE: Text- What is it that R dislikes about Republican Pres cand
           'V161098', # PRE: Text- What does R like about Democratic party
           'V161101', # PRE: Text- What does R dislike about the Democratic party
           'V161104', # PRE: Text- What does R like about Republican party
           'V161106'  # PRE: Text- What does R dislike about the Republican party
          ]

# Rescale ideology from 0-6 to 0-2 
pids = {0: 0, 1: 0, 2: 1, 3: 1, 4: 1, 5: 2, 6: 2, '': np.nan} # 0, 1 = dem; 2, 3, 4 = ind; 5, 6 = rep
df['party_id'] = df['leftright'].fillna('').apply(lambda x: pids[x]) 

In [ ]:
# number of respondents per question
for q in text_qs:
    sub = df.dropna(subset=q)
    print(f'Question {q}: {len(sub)} respondents')

# 2. Clean Text fields

In [19]:
### Load language model
nlp = spacy.load('en_core_web_trf')
contextualSpellCheck.add_to_pipe(nlp) # use context to correct spelling

# Note: spell check is very memory intensive. 
# We will do one pass cleaning text (with spellcheck), but save result as text (for memory)
# In future notebooks, we'll  reload the nlp model (without adding spell check) and re-convert cleaned text everything to Spacy docs

In [9]:
# simple text cleaning to replace line break characters
def clean_text(x):
    x = x.replace('//', '. ')
    x = x.replace('\\', '. ')
    x = x.replace('.', '. ')
    
    return x.lower()

In [20]:
#### Pass 1: Clean text with spell check
# clean each text question, this may take some time
for q in text_qs:
    print(f'  Processing question {q}')
    df[f'{q}_clean'] = df[q].fillna('').apply(lambda x: nlp(clean_text(x)).text)
    
print('All text cleaned with spell check.')

  Processing question V161069
  Processing question V161072
  Processing question V161075
  Processing question V161078
  Processing question V161098
  Processing question V161101
  Processing question V161104
  Processing question V161106
All text cleaned with spell check.


In [22]:
# save cleaned data to file
df.to_csv('data/ANES_2016_CLEANED.txt', sep='\t', index=False)